# Sprint 4 — Sesión Teórica Consolidada (70 min)
## Data Journey → Funnels → Cohorts con SQL en Google Colab

**Modalidad:** explicación guiada + consultas SQL cortas  
**Motor:** DuckDB en Google Colab  
**Datos:** archivos `.parquet` cargados desde GitHub, carpeta `/datasets`  
**Prefijo de archivos:** `sprint04_`

> Esta sesión consolida los contenidos teóricos de las sesiones originales del Sprint 4: Data Journey, CTEs, funnels, cohorts y lectura analítica de eventos.

<div style="text-align: center">
    <img src="https://raw.githubusercontent.com/ljpiere/tpdata_python/main/images/w1s1_2.png" width="400">
</div>

## Estructura y timeboxing

| Tiempo | Bloque | Modalidad | Contenido |
|---:|---|---|---|
| 0–5 | Apertura | Plenaria | Objetivo, contexto y flujo de trabajo en Colab |
| 5–15 | Bloque 0 | Coding Together | Cargar tablas `.parquet` desde GitHub y crear tablas SQL |
| 15–28 | Bloque 1 | Teórico + SQL | Data Journey, eventos y validaciones mínimas |
| 28–40 | Bloque 2 | Teórico + SQL | CTEs para ordenar transformaciones |
| 40–55 | Bloque 3 | Coding Together | Funnel de conversión y drop-off |
| 55–63 | Bloque 4 | Demo guiada | Cohorts y retención |
| 63–70 | Cierre | Plenaria | Preguntas de validación, takeaways y canales de ayuda |

## Objetivos de aprendizaje

Al finalizar la sesión, el/la estudiante podrá:

1. Explicar qué es un **Data Journey** y cómo se representa con eventos.
2. Identificar campos clave para reconstruir un journey: `user_id`, `event_ts`, `event_name`, `session_id`.
3. Usar **CTEs** para dividir una consulta compleja en pasos claros.
4. Construir un **funnel de conversión** con SQL.
5. Interpretar métricas de conversión, drop-off y retención por cohort.

---
# Bloque 0 · Preparación del entorno en Google Colab

En esta versión trabajaremos en **Google Colab**. Las tablas se leerán desde GitHub usando URLs raw con esta estructura:

```text
https://github.com/ljpiere/tpdata_course/raw/refs/heads/main/new_DA/DA_tutor/datasets/sprint04_*.parquet
```

Archivos usados en clase:

- `sprint04_users.parquet`
- `sprint04_events.parquet`
- `sprint04_funnel_sessions.parquet`
- `sprint04_experiments.parquet`

> Antes de ejecutar esta celda en clase, sube los archivos `.parquet` a la carpeta `new_DA/DA_tutor/datasets/` del repo.

In [ ]:
# Instalación e importación de librerías
!pip install -q duckdb pandas pyarrow requests

import io
import requests
import pandas as pd
import duckdb

BASE_URL = "https://github.com/ljpiere/tpdata_course/raw/refs/heads/main/new_DA/DA_tutor/datasets"

DATASETS = {
    "users": "sprint04_users.parquet",
    "events": "sprint04_events.parquet",
    "funnel_sessions": "sprint04_funnel_sessions.parquet",
    "experiments": "sprint04_experiments.parquet",
}

def read_parquet_from_github(file_name: str) -> pd.DataFrame:
    url = f"{BASE_URL}/{file_name}"
    response = requests.get(url)
    response.raise_for_status()
    return pd.read_parquet(io.BytesIO(response.content))

# Cargar archivos parquet desde GitHub
users = read_parquet_from_github(DATASETS["users"])
events = read_parquet_from_github(DATASETS["events"])
funnel_sessions = read_parquet_from_github(DATASETS["funnel_sessions"])
experiments = read_parquet_from_github(DATASETS["experiments"])

# Crear conexión DuckDB en memoria
con = duckdb.connect(database=":memory:")

# Registrar DataFrames como tablas SQL
con.register("users_df", users)
con.register("events_df", events)
con.register("funnel_sessions_df", funnel_sessions)
con.register("experiments_df", experiments)

con.execute("CREATE OR REPLACE TABLE users AS SELECT * FROM users_df")
con.execute("CREATE OR REPLACE TABLE events AS SELECT * FROM events_df")
con.execute("CREATE OR REPLACE TABLE funnel_sessions AS SELECT * FROM funnel_sessions_df")
con.execute("CREATE OR REPLACE TABLE experiments AS SELECT * FROM experiments_df")

def sql(query: str) -> pd.DataFrame:
    """Ejecuta una consulta SQL en DuckDB y devuelve un DataFrame."""
    return con.execute(query).df()

print("Tablas creadas en DuckDB:", ", ".join(DATASETS.keys()))

## 0.1 Verificación rápida de tablas

La primera consulta valida que Colab logró descargar los archivos y que DuckDB creó las tablas correctamente.

In [ ]:
sql("""
SELECT 'users' AS table_name, COUNT(*) AS rows FROM users
UNION ALL
SELECT 'events' AS table_name, COUNT(*) AS rows FROM events
UNION ALL
SELECT 'funnel_sessions' AS table_name, COUNT(*) AS rows FROM funnel_sessions
UNION ALL
SELECT 'experiments' AS table_name, COUNT(*) AS rows FROM experiments;
""")

---
# Bloque 1 · ¿Qué es el Data Journey? (13 min)

Un **Data Journey** representa el recorrido de un usuario dentro de un producto digital. En vez de mirar únicamente ventas o conversiones finales, se observan las etapas intermedias:

| Etapa | Evento posible | Pregunta de negocio |
|---|---|---|
| Registro | `signup` | ¿Cuántos usuarios nuevos llegan? |
| Activación | `app_open` | ¿Cuántos usuarios empiezan a usar el producto? |
| Interés | `view_product` | ¿Cuántos exploran productos? |
| Intención | `add_to_cart` | ¿Cuántos muestran intención de compra? |
| Checkout | `begin_checkout` | ¿Cuántos inician pago? |
| Conversión | `purchase` | ¿Cuántos compran? |

La tabla central para reconstruir este recorrido es `events`.

In [ ]:
sql("""
SELECT *
FROM events
ORDER BY user_id, event_ts
LIMIT 10;
""")

## 1.1 Campos clave para reconstruir journeys

| Campo | Uso analítico |
|---|---|
| `user_id` | Identifica al usuario |
| `session_id` | Agrupa eventos de una misma sesión |
| `event_ts` | Permite ordenar cronológicamente el recorrido |
| `event_name` | Define la acción del usuario |
| `platform`, `device`, `page` | Permiten segmentar el análisis |

Antes de calcular funnels o cohorts, se valida la calidad mínima de los eventos.

### Ejercicio 1.1 — Eventos duplicados exactos

**Objetivo:** detectar eventos repetidos que podrían inflar conteos de actividad o conversión.

In [ ]:
sql("""
SELECT
    user_id,
    session_id,
    event_ts,
    event_name,
    COUNT(*) AS n
FROM events
GROUP BY user_id, session_id, event_ts, event_name
HAVING COUNT(*) > 1
ORDER BY n DESC;
""")

### Ejercicio 1.2 — Eventos anteriores al registro

**Objetivo:** verificar coherencia temporal entre la tabla `users` y la tabla `events`.

In [ ]:
sql("""
SELECT
    e.user_id,
    u.signup_ts,
    e.event_ts,
    e.event_name
FROM events e
JOIN users u
    ON e.user_id = u.user_id
WHERE e.event_ts < u.signup_ts
ORDER BY e.user_id, e.event_ts;
""")

---
# Bloque 2 · CTEs para ordenar transformaciones (12 min)

Una **CTE** (*Common Table Expression*) es una consulta temporal con nombre. Sirve para dividir una consulta compleja en pasos legibles.

Estructura general:

```sql
WITH nombre_cte AS (
    SELECT ...
    FROM ...
)
SELECT *
FROM nombre_cte;
```

En análisis de journeys, las CTEs ayudan a construir etapas: primero limpiar eventos, luego resumir por usuario, luego calcular conversiones.

### Ejercicio 2.1 — Primera CTE: eventos válidos

Filtramos eventos que ocurren después del registro del usuario.

In [ ]:
sql("""
WITH eventos_validos AS (
    SELECT
        e.event_id,
        e.user_id,
        e.session_id,
        e.event_ts,
        e.event_name
    FROM events e
    JOIN users u
        ON e.user_id = u.user_id
    WHERE e.event_ts >= u.signup_ts
)
SELECT *
FROM eventos_validos
ORDER BY user_id, event_ts
LIMIT 10;
""")

### Ejercicio 2.2 — CTE encadenada: primer evento por usuario

Usamos una segunda CTE para resumir el primer evento válido de cada usuario.

In [ ]:
sql("""
WITH eventos_validos AS (
    SELECT e.*
    FROM events e
    JOIN users u
        ON e.user_id = u.user_id
    WHERE e.event_ts >= u.signup_ts
),
primer_evento AS (
    SELECT
        user_id,
        MIN(event_ts) AS first_event_ts
    FROM eventos_validos
    GROUP BY user_id
)
SELECT
    p.user_id,
    p.first_event_ts,
    u.channel,
    u.segment
FROM primer_evento p
JOIN users u
    ON p.user_id = u.user_id
ORDER BY p.user_id
LIMIT 10;
""")

---
# Bloque 3 · Funnel de conversión (15 min)

Un **funnel** mide cuántos usuarios pasan por etapas sucesivas. En este caso usamos eventos:

`signup → view_product → add_to_cart → begin_checkout → purchase`

La clave es resumir cada etapa a nivel de usuario y luego contar usuarios por etapa.

### Ejercicio 3.1 — Funnel general con CTEs

In [ ]:
sql("""
WITH eventos_usuario AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_name = 'signup' THEN 1 ELSE 0 END) AS has_signup,
        MAX(CASE WHEN event_name = 'view_product' THEN 1 ELSE 0 END) AS has_view_product,
        MAX(CASE WHEN event_name = 'add_to_cart' THEN 1 ELSE 0 END) AS has_add_to_cart,
        MAX(CASE WHEN event_name = 'begin_checkout' THEN 1 ELSE 0 END) AS has_begin_checkout,
        MAX(CASE WHEN event_name = 'purchase' THEN 1 ELSE 0 END) AS has_purchase
    FROM events
    GROUP BY user_id
)
SELECT
    COUNT(*) AS users_total,
    SUM(has_signup) AS signup_users,
    SUM(has_view_product) AS view_product_users,
    SUM(has_add_to_cart) AS add_to_cart_users,
    SUM(has_begin_checkout) AS begin_checkout_users,
    SUM(has_purchase) AS purchase_users
FROM eventos_usuario;
""")

### Ejercicio 3.2 — Funnel con tasas y drop-off

La lectura de negocio no se queda en conteos. También se calculan tasas de conversión y pérdida entre etapas.

In [ ]:
sql("""
WITH eventos_usuario AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_name = 'signup' THEN 1 ELSE 0 END) AS has_signup,
        MAX(CASE WHEN event_name = 'view_product' THEN 1 ELSE 0 END) AS has_view_product,
        MAX(CASE WHEN event_name = 'add_to_cart' THEN 1 ELSE 0 END) AS has_add_to_cart,
        MAX(CASE WHEN event_name = 'begin_checkout' THEN 1 ELSE 0 END) AS has_begin_checkout,
        MAX(CASE WHEN event_name = 'purchase' THEN 1 ELSE 0 END) AS has_purchase
    FROM events
    GROUP BY user_id
),
conteos AS (
    SELECT
        SUM(has_signup) AS signup_users,
        SUM(has_view_product) AS view_product_users,
        SUM(has_add_to_cart) AS add_to_cart_users,
        SUM(has_begin_checkout) AS begin_checkout_users,
        SUM(has_purchase) AS purchase_users
    FROM eventos_usuario
)
SELECT
    signup_users,
    view_product_users,
    add_to_cart_users,
    begin_checkout_users,
    purchase_users,
    ROUND(view_product_users * 1.0 / signup_users, 3) AS signup_to_view_rate,
    ROUND(add_to_cart_users * 1.0 / view_product_users, 3) AS view_to_cart_rate,
    ROUND(begin_checkout_users * 1.0 / add_to_cart_users, 3) AS cart_to_checkout_rate,
    ROUND(purchase_users * 1.0 / begin_checkout_users, 3) AS checkout_to_purchase_rate,
    signup_users - purchase_users AS total_drop_off
FROM conteos;
""")

---
# Bloque 4 · Cohorts y retención (8 min)

Un **cohort** agrupa usuarios por una característica común. En productos digitales, una cohorte frecuente es el mes de registro.

La retención responde preguntas como:

- ¿Los usuarios registrados en enero volvieron en la semana 1, 2 o 3?
- ¿Qué cohort mantiene más actividad?
- ¿La retención cambia por canal, país o segmento?

### Ejercicio 4.1 — Retención semanal por cohorte mensual

In [ ]:
sql("""
WITH actividad AS (
    SELECT
        u.user_id,
        DATE_TRUNC('month', u.signup_ts) AS cohort_month,
        DATE_DIFF('week', CAST(u.signup_ts AS DATE), CAST(e.event_ts AS DATE)) AS week_number
    FROM users u
    JOIN events e
        ON u.user_id = e.user_id
    WHERE e.event_name = 'app_open'
      AND e.event_ts >= u.signup_ts
),
retencion AS (
    SELECT
        cohort_month,
        week_number,
        COUNT(DISTINCT user_id) AS retained_users
    FROM actividad
    WHERE week_number BETWEEN 0 AND 4
    GROUP BY cohort_month, week_number
)
SELECT *
FROM retencion
ORDER BY cohort_month, week_number;
""")

---
# Cierre · Validación de conocimiento y takeaways (7 min)

## Preguntas de validación para el estudiante

1. ¿Qué campo permite ordenar correctamente los eventos de un journey?
2. ¿Por qué no basta con contar compras para entender el comportamiento del usuario?
3. ¿Qué problema resuelve una CTE en una consulta compleja?
4. ¿Cuál es la diferencia entre conversión y drop-off?
5. ¿Qué pregunta responde un análisis de cohorts que un funnel no responde?

## Respuestas esperadas / criterios

1. `event_ts`, porque permite reconstruir la secuencia temporal.
2. Porque la compra es solo la etapa final; necesitamos conocer dónde se pierden los usuarios.
3. Divide la lógica en pasos nombrados, más legibles y fáciles de validar.
4. Conversión mide avance entre etapas; drop-off mide pérdida entre etapas.
5. Cohorts mide comportamiento en el tiempo de grupos de usuarios, especialmente retención.

## Takeaways de la sesión

- Un journey traduce eventos individuales en una historia analítica del usuario.
- La calidad de datos debe validarse antes de calcular funnels o cohorts.
- Las CTEs son una herramienta de claridad: permiten construir análisis complejos paso a paso.
- Un funnel muestra avance y pérdida entre etapas.
- Un cohort permite analizar retención y evolución en el tiempo.

### ¿Necesitas ayuda?

Recuerda los canales de ayuda disponibles:

- `DATA-CO-LEARNING`: revisa los horarios de atención en la guía del estudiante. Hay horarios de apoyo para dudas puntuales.
- `DA_CONSULTA`: publica preguntas sobre el contenido, ejercicios o errores de SQL.
- Sesiones de acompañamiento: lleva el error exacto, la consulta SQL y una captura del resultado esperado vs. obtenido.

## Siguientes pasos

En la sesión práctica vamos a usar estas mismas tablas para construir funnels por variante A/B, simular mejoras en checkout y traducir cambios de conversión a impacto de revenue.